In [1]:
import os
import re
import random
import traceback
from datetime import timedelta

import numpy as np
import mne
import gc
import shutil

# ==========================================================================
# 0. CONSTANS AND SEIZURE INFORMATION FUNCTION
# ==========================================================================
CHANNELS_18 = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ',  'CZ-PZ'
]

def select_channels(raw, channel_list):
    rename_dict = {}
    if 'T8-P8-0' in raw.ch_names:
        raw.rename_channels({'T8-P8-0': 'T8-P8'})
    for ch in raw.ch_names:
        clean_name = ch.strip().replace('EEG ', '').upper()
        rename_dict[ch] = clean_name
    raw.rename_channels(rename_dict)

    existing_channels = [ch for ch in channel_list if ch in raw.ch_names]
    print(f'Threre are {len(existing_channels)} channels: ', existing_channels)
    if len(existing_channels) > 0:
        raw.pick(existing_channels)
    return raw


def load_seizures_from_summary(summary_path, edf_filename):
    """Read seizure information (onset, offset) from file summary.txt."""
    seizures = []
    if not os.path.exists(summary_path):
        return seizures

    with open(summary_path, 'r', encoding='latin-1') as f:
        content = f.read()

    file_blocks = content.split('File Name:')
    for block in file_blocks:
        if edf_filename.lower() in block.lower():
            num_seizures_match = re.search(r'Number of Seizures in File:\s*(\d+)', block)
            if not num_seizures_match or int(num_seizures_match.group(1)) == 0:
                return seizures

            onsets = re.findall(r'Seizure\s*(?:\d+)?\s*Start Time:\s*(\d+)\s*seconds', block)
            offsets = re.findall(r'Seizure\s*(?:\d+)?\s*End Time:\s*(\d+)\s*seconds', block)

            for on, off in zip(onsets, offsets):
                seizures.append((float(on), float(off)))
            break

    return seizures

In [2]:
# ==========================================================================
# 0.1 CHECK NUMBER OF SEIZURES
# ==========================================================================
import re
import os
from collections import defaultdict

summary_dir = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'

seizure_counts = defaultdict(int)

for folder in os.listdir(summary_dir):
    folder_dir = os.path.join(summary_dir, folder)

    if not os.path.isdir(folder_dir):
        continue

    patient = folder.upper()

    for filename in os.listdir(folder_dir):
        if not filename.endswith('-summary.txt'):
            continue

        filepath = os.path.join(folder_dir, filename)

        with open(filepath, 'r') as f:
            text = f.read()

        matches = re.findall(r'Number of Seizures in File:\s*(\d+)', text)
        n_seizures = sum(int(x) for x in matches)

        seizure_counts[patient] = n_seizures

        break

for patient, n in sorted(seizure_counts.items()):
    print(f'{patient}: {n} seizures')

CHB01: 7 seizures
CHB02: 3 seizures
CHB03: 7 seizures
CHB04: 4 seizures
CHB05: 5 seizures
CHB06: 10 seizures
CHB07: 3 seizures
CHB08: 5 seizures
CHB09: 4 seizures
CHB10: 7 seizures
CHB11: 3 seizures
CHB12: 40 seizures
CHB13: 12 seizures
CHB14: 8 seizures
CHB15: 20 seizures
CHB16: 10 seizures
CHB17: 3 seizures
CHB18: 6 seizures
CHB19: 3 seizures
CHB20: 8 seizures
CHB21: 4 seizures
CHB22: 3 seizures
CHB23: 7 seizures
CHB24: 16 seizures


In [4]:
# ==========================================================================
# 1. INTERVAL LENGTHS AND INPUT, OUTPUT PATHs
# ==========================================================================
PREICTAL_SEC          = 30 * 60        # Preictal restricted at 30 minutes
INTERICTAL_DIST_SEC   = 4 * 60 * 60    # Interictal  must be 4 HOURS away from every onsets and offsets
POSTICTAL_SEC  = 60 * 60        # Two consecutive seizures must be 60 minutes away from each other
CONTINUOUS_TOLERANCE_SEC = 10.0         # two continuous files must be <=10 minutes away from each other

EPOCH_LEN_SEC = 10                     # epoch length

#Kaggle input path
LOCAL_DATA = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'

# output path
SAVE_ROOT  = '/kaggle/working/CHB-MIT_Segmentation'

In [5]:
# ==========================================================================
# 2. BUILD TIMELINE FOR ONE PATIENT
# ==========================================================================
def _discover_edf_files(subject_dir):
    return sorted([f for f in os.listdir(subject_dir) if f.lower().endswith('.edf')])


def _read_file_infos(subject_dir, subject, edf_files):
    summary_path = os.path.join(subject_dir, f"{subject}-summary.txt")
    file_infos = []
    for f in edf_files:
        raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
        raw = select_channels(raw, CHANNELS_18)

        meas_date = raw.info['meas_date']
        if meas_date is None:
            print(f"[WARN] {f}: No meas_date -> skip this file")
            continuefno

        duration = raw.times[-1] + (1.0 / raw.info['sfreq'])
        seizures = load_seizures_from_summary(summary_path, f)
        file_infos.append({
            'file': f,
            'raw': raw,
            'start': meas_date,
            'duration': duration,
            'end': meas_date + timedelta(seconds=duration),
            'seizures': seizures,
        })

    file_infos.sort(key=lambda x: x['start'])
    return file_infos


def _group_into_blocks(file_infos):
    if not file_infos:
        return []

    blocks = [[file_infos[0]]]
    for prev, cur in zip(file_infos, file_infos[1:]):
        gap = (cur['start'] - prev['end']).total_seconds()
        if 0 <= gap <= CONTINUOUS_TOLERANCE_SEC:
            blocks[-1].append(cur)
        else:
            blocks.append([cur])
    return blocks


def _concat_block(block_files):
    raws = []
    for fi in block_files:
        raw = fi['raw'].copy()
        raw.load_data()
        raw.notch_filter(freqs=60.0)
        raws.append(raw)
    raw_concat = raws[0] if len(raws) == 1 else mne.concatenate_raws(raws)
    block_start = block_files[0]['start']
    return raw_concat, block_start


def build_timeline(subject_dir, subject):
    edf_files = _discover_edf_files(subject_dir)
    file_infos = _read_file_infos(subject_dir, subject, edf_files)
    grouped = _group_into_blocks(file_infos)

    blocks = []
    for block_files in grouped:
        raw_concat, block_start = _concat_block(block_files)
        blocks.append({
            'raw': raw_concat,
            'start': block_start,
            'duration': raw_concat.times[-1] + (1.0 / raw_concat.info['sfreq']),
            'files': block_files,
        })
    return blocks

In [6]:
# ==========================================================================
# 3. COLLECT EVERY SEIZURES OF A PATIENT AND NUMBERING SEIZURES FOR LOSO EVALUATION (LEAVE-ONE SEIZURE-OUT)
# ==========================================================================
def collect_seizures(blocks):
    all_seizures = []
    for b_idx, block in enumerate(blocks):
        block_start = block['start']
        cum = 0.0
        for finfo in block['files']:
            for (onset, offset) in finfo['seizures']:
                all_seizures.append({
                    'block_idx': b_idx,
                    'onset_in_block': cum + onset,
                    'offset_in_block': cum + offset,
                    'onset_abs': block_start + timedelta(seconds=cum + onset),
                    'offset_abs': block_start + timedelta(seconds=cum + offset),
                })
            cum += finfo['duration']

    all_seizures.sort(key=lambda s: s['onset_abs'])
    for i, s in enumerate(all_seizures):
        s['seizure_id'] = i + 1
    return all_seizures

In [34]:
# ==========================================================================
# 4. GET PREICTAL INTERVAL
# ==========================================================================
def is_preictal_valid(seizure, prev):
    if prev == None:
        return True
    gap_to_prev = (seizure['onset_abs'] - prev['offset_abs']).total_seconds()
    print(seizure['onset_abs'],prev['offset_abs'])
    if gap_to_prev < (POSTICTAL_SEC + EPOCH_LEN_SEC):
        print(gap_to_prev)
        return False
    else:
        return True

def get_seizure_windows(seizure, block, all_seizures):
    idx = seizure['seizure_id']
    
    if idx>1:
        prev = next(x for x in all_seizures if x["seizure_id"] == idx-1)
    else:
        prev=None
        
    onset = seizure['onset_in_block']
    windows = {}
    
    valid = is_preictal_valid(seizure, prev)
    
    if valid:
        if prev==None:
            pre_start = max(onset - PREICTAL_SEC, 0)
        elif prev['block_idx'] == seizure['block_idx']:
            pre_start = max(onset - PREICTAL_SEC, prev['offset_in_block'] + (POSTICTAL_SEC))
        else:
            pre_start = max(onset - PREICTAL_SEC, 0)
            
        windows['preictal'] = (pre_start, onset)
    else:
        windows['preictal'] = None

    return windows

In [42]:
def get_lead_seizures(all_seizures, rec_start, min_gap_sec=INTERICTAL_DIST_SEC):
   
    seizures = sorted(all_seizures, key=lambda s: s['onset_abs'])
    leads = []
    for i, s in enumerate(seizures):
        if i == 0:
            gap_prev = (s['onset_abs'] - rec_start).total_seconds()
        else:
            gap_prev = (s['onset_abs'] - seizures[i - 1]['offset_abs']).total_seconds()
        gap_prev_ok = gap_prev >= min_gap_sec

        gap_next_ok = True
        if i < len(seizures) - 1:
            gap_next = (seizures[i + 1]['onset_abs'] - s['offset_abs']).total_seconds()
            gap_next_ok = gap_next >= min_gap_sec

        if gap_prev_ok and gap_next_ok:
            leads.append((s['seizure_id'], s))
    return seizures, leads

def get_interictal_windows(block, all_seizures):
    block_start = block['start']
    block_duration = block['duration']
    block_end = block_start + timedelta(seconds=block_duration)

    excluded = []
    for s in all_seizures:
        excl_start_abs = s['onset_abs'] - timedelta(seconds=INTERICTAL_DIST_SEC)
        excl_end_abs = s['offset_abs'] + timedelta(seconds=INTERICTAL_DIST_SEC)
        seg_start_abs = max(excl_start_abs, block_start)
        seg_end_abs = min(excl_end_abs, block_end)
        if seg_start_abs < seg_end_abs:
            excluded.append((
                (seg_start_abs - block_start).total_seconds(),
                (seg_end_abs - block_start).total_seconds(),
            ))

    excluded.sort()
    merged = []
    for s0, e0 in excluded:
        if merged and s0 <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e0))
        else:
            merged.append((s0, e0))

    windows, cursor = [], 0.0
    for s0, e0 in merged:
        if cursor < s0:
            windows.append((cursor, s0))
        cursor = max(cursor, e0)
    if cursor < block_duration:
        windows.append((cursor, block_duration))
    return windows

def get_interictal_windows_by_seizure(block, all_seizures, rec_start):
    seizures, leads = get_lead_seizures(all_seizures, rec_start)
    print(f'{len(leads)} seizures for interictal segmentation')
    base_windows = get_interictal_windows(block, seizures)
    block_start = block['start']

    result = {}
    for idx, s in leads:
        lo = seizures[idx - 1]['offset_abs'] if idx > 0 else rec_start
        hi = s['onset_abs']
        for w0, w1 in base_windows:
            a0 = block_start + timedelta(seconds=w0)
            a1 = block_start + timedelta(seconds=w1)
            o0, o1 = max(a0, lo), min(a1, hi)
            if o0 < o1:
                result.setdefault(idx, []).append((
                    (o0 - block_start).total_seconds(),
                    (o1 - block_start).total_seconds()))
    return result

In [54]:
get_interictal_windows_by_seizure(blocks[], all_seizures, min(b['start'] for b in blocks))

IndexError: list index out of range

In [51]:
len(blocks)

7

In [8]:
blocks = build_timeline('/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0/chb03', 'chb03')
all_seizures = collect_seizures(blocks)

/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)


Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']


/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/187617

Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-

/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/187617

Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-

/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/187617

Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Threre are 18 channels:  ['FP1-F7', 'F7-

/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/1876176323.py:12: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_58/187617

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1691 samples (6.605 s)

Reading 0 ... 921599  =      0.000 ...  3599.996 secs...
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50

In [31]:
# GET CONTINUOUS WINDOW FOR FAR/H CALCULATION
def get_testing_windows(block, block_idx, blocks, all_seizures):
    sz_in_block = [sz for sz in all_seizures if (sz['block_idx']==block_idx)]
    block_raw = block['raw']
    windows = []
    previous_sz = None
    if len(sz_in_block)>0:
        for idx,sz in enumerate(sz_in_block):
            if idx==0:
                windows.append((0, sz['onset_in_block']))
            elif previous_sz is not None:
                windows.append((previous_sz['offset_in_block'], sz['onset_in_block']))
            previous_sz = sz
        windows.append((sz['offset_in_block'], block_raw.times[-1] + (1/block_raw.info['sfreq'])))  
        
        return sz_in_block, windows
    else:
        return sz_in_block, [(0, block_raw.times[-1] + (1/block_raw.info['sfreq']))]

In [32]:
# ==========================================================================
# 7. PROCESSING PIPELINE FOR ONE PATIENT 
# ==========================================================================
def process_subject(subject):
    subject_dir = os.path.join(LOCAL_DATA, subject)
    save_dir = os.path.join(SAVE_ROOT, subject)

    print(f"\n--- PROCESSING: {subject.upper()} ---")
    blocks = build_timeline(subject_dir, subject)
    all_seizures = collect_seizures(blocks)

    if len(all_seizures) == 0:
        print(f"{subject}: No seizure, skip")
        return

    print(f"{subject}: Total {len(all_seizures)} seizures (1..{len(all_seizures)})")

    # ---- PHASE 1: GET PREICTAL EPOCHS ----
    print('PHASE 1')
    training_dir = os.path.join(save_dir, 'Training')
    class_dir = os.path.join(training_dir, 'preictal')
    num_pre_epochs = 0
    for s in all_seizures:
        block = blocks[s['block_idx']]
        raw = block['raw']
        windows = get_seizure_windows(s, block, all_seizures)
        sid = s['seizure_id']

        win = windows['preictal']
        if win is None:
            print(f"  Seizure number {sid:02d}: does not contain enough preictal data (skip this seizure)")
            continue

        epochs = crop_to_epochs(raw, win[0], win[1])
        epochs.get_data()
        save_fold_npy(epochs, sid, class_dir)
        num_pre_epochs += len(epochs)
        del epochs
        print(f"  Seizure number {sid:02d}: Preictal OK")
    print(f"Total preictal epochs {num_pre_epochs}")

    # ---- PHASE 2: GET INTERICTAL EPOCHS (lưu theo cơn) ----
    print('PHASE 2')
    interictal_root = os.path.join(training_dir, 'interictal')
    batch_size = 1800
    batch_idx_per_seizure = {}   # mỗi cơn đếm batch riêng
    num_inter_epochs = {}        # số epoch interictal theo cơn
    rec_start = min(b['start'] for b in blocks)
    for block_idx, block in enumerate(blocks):
        print(f"Block: {block_idx} tu {block['files'][0]['file']} den {block['files'][-1]['file']}")
        print(f"Block duration: {block['duration']}")
        windows_by_seizure = get_interictal_windows_by_seizure(block, all_seizures, rec_start)
        for sz_idx, windows in windows_by_seizure.items():
            class_dir = os.path.join(interictal_root, f"seizure_{sz_idx:02d}")
            os.makedirs(class_dir, exist_ok=True)
            batch_idx = batch_idx_per_seizure.get(sz_idx, 0)
    
            for (w0, w1) in windows:
                epochs = crop_to_epochs(block['raw'], w0, w1)
                if epochs is not None and len(epochs) > 0:
                    num_inter_epochs[sz_idx] = num_inter_epochs.get(sz_idx, 0) + len(epochs)
                    for i in range(0, len(epochs), batch_size):
                        print(f"Patient: {subject} | Seizure: {sz_idx}")
                        print(f"Epochs: {len(epochs)}")
                        print(f"Batch: {i}:{min(i + batch_size, len(epochs))}")
                        batch = epochs[i:i + batch_size].get_data(copy=False)
                        save_fold_npy(batch, batch_idx, class_dir)
                        batch_idx += 1
                        del batch
                del epochs
    
            batch_idx_per_seizure[sz_idx] = batch_idx
    # ---- PHASE 3: SEGMENT CONTINUOUS DATA TO CALCULATE FAR/H ----
    print('PHASE 3')
    testing_dir = os.path.join(save_dir, 'testing')
    previous_epochs = None
    batch_size = 1800
    for block_idx, block in enumerate(blocks):
        print(f"Block: {block_idx} from {block['files'][0]['file']} to {block['files'][-1]['file']}")
        raw = block['raw']
        sz_in_block, windows = get_testing_windows(block, block_idx, blocks, all_seizures)
        if len(sz_in_block)>0:
            for (idx, sz), window in zip(enumerate(sz_in_block), windows[:len(windows)-1]):
                sid = sz['seizure_id']
                print('seizure ID: ', sid)
                testing_dir_sz = os.path.join(testing_dir, str(sid))
                epochs = crop_to_epochs(raw, window[0], window[1])
                if epochs is not None:
                    batch_idx = 0
                    print(f'Epoch size: {len(epochs)}')
                    for i in range(0, len(epochs), batch_size):
                        print(f"Batch: {i}:{min(i + batch_size, len(epochs))}")
                        batch = epochs[i:i + batch_size].get_data(copy=False)
                        save_fold_npy(batch, batch_idx, testing_dir_sz)
                        batch_idx += 1
                        del batch
                    del epochs

'# ==========================================================================\n# 7. PROCESSING PIPELINE FOR ONE PATIENT \n# ==========================================================================\ndef process_subject(subject):\n    subject_dir = os.path.join(LOCAL_DATA, subject)\n    save_dir = os.path.join(SAVE_ROOT, subject)\n\n    print(f"\n--- PROCESSING: {subject.upper()} ---")\n    blocks = build_timeline(subject_dir, subject)\n    all_seizures = collect_seizures(blocks)\n\n    if len(all_seizures) == 0:\n        print(f"{subject}: No seizure, skip")\n        return\n\n    print(f"{subject}: Total {len(all_seizures)} seizures (1..{len(all_seizures)})")\n\n    # ---- PHASE 1: GET PREICTAL EPOCHS ----\n    print(\'PHASE 1\')\n    training_dir = os.path.join(save_dir, \'Training\')\n    class_dir = os.path.join(training_dir, \'preictal\')\n    num_pre_epochs = 0\n    for s in all_seizures:\n        block = blocks[s[\'block_idx\']]\n        raw = block[\'raw\']\n        windows, v

In [41]:
blocks

[{'raw': <RawEDF | chb03_01.edf, 18 x 18433536 (72006.0 s), ~2.47 GiB, data loaded>,
  'start': datetime.datetime(2075, 9, 2, 13, 23, 36, tzinfo=datetime.timezone.utc),
  'duration': np.float64(72006.0),
  'files': [{'file': 'chb03_01.edf',
    'raw': <RawEDF | chb03_01.edf, 18 x 921600 (3600.0 s), ~18 KiB, data not loaded>,
    'start': datetime.datetime(2075, 9, 2, 13, 23, 36, tzinfo=datetime.timezone.utc),
    'duration': np.float64(3600.0),
    'end': datetime.datetime(2075, 9, 2, 14, 23, 36, tzinfo=datetime.timezone.utc),
    'seizures': [(362.0, 414.0)]},
   {'file': 'chb03_02.edf',
    'raw': <RawEDF | chb03_02.edf, 18 x 921600 (3600.0 s), ~18 KiB, data not loaded>,
    'start': datetime.datetime(2075, 9, 2, 14, 23, 39, tzinfo=datetime.timezone.utc),
    'duration': np.float64(3600.0),
    'end': datetime.datetime(2075, 9, 2, 15, 23, 39, tzinfo=datetime.timezone.utc),
    'seizures': [(731.0, 796.0)]},
   {'file': 'chb03_03.edf',
    'raw': <RawEDF | chb03_03.edf, 18 x 921600 (3